# Notebook 1: Get the pages ready

This notebook downloads the competition's pages (all three collections) from S3, and prepares them for
training:

- it sets aside the team's **dev pages** (55 pages from whole documents, listed in `dev_ids.csv`),
  which are never trained on and are what every run is scored on;
- it cuts every **survey spread** (an open notebook, two pages side by side) at the fold, so the
  model reads one page at a time, and splits the label to match where it can;
- it takes the `Postmark` line out of the Kade labels it trains on (the postmark is on the back of
  the card, so it isn't in the image).

Run it once. About 15 to 20 minutes, mostly downloading about 2.5 GB of images.
Run the cells from top to bottom with **Shift + Enter**. Kernel: **conda_pytorch_p310** (or another `conda_pytorch` kernel, not `conda_python3`).

## Step 1. Set up the folders

Nothing to change here.

In [ ]:
import importlib.util
import os
import subprocess
import sys

# Which kernel is this? It should be conda_pytorch_p310 (or another conda_pytorch kernel).
print("Kernel environment:", os.path.basename(os.path.dirname(os.path.dirname(sys.executable))))
missing = [pkg for module, pkg in (("PIL", "pillow"), ("pandas", "pandas"))
           if importlib.util.find_spec(module) is None]
if missing:
    print("Installing", ", ".join(missing), "into this kernel ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# Everything this project downloads or creates lives here, on the notebook
# instance's storage volume, so it survives Stop and Start.
WORK = "/home/ec2-user/SageMaker/churro-work"
DATA = os.path.join(WORK, "data")     # page images and lists of pages
RUNS = os.path.join(WORK, "runs")     # one folder per training run
SCRIPTS = os.path.abspath("scripts")  # the scripts folder next to this notebook
for folder in (WORK, DATA, RUNS):
    os.makedirs(folder, exist_ok=True)
sys.path.insert(0, SCRIPTS)
os.environ["HF_HOME"] = os.path.join(WORK, "hf_cache")   # the 7.5 GB model download goes here

# Your bucket name is saved here, outside the repository, so it's typed once and
# updating the code (git pull) never touches it.
import json
SETTINGS = os.path.join(WORK, "my_settings.json")
def saved_setting(name, default=""):
    try:
        return json.load(open(SETTINGS)).get(name, default)
    except (OSError, ValueError):
        return default
def save_setting(name, value):
    try:
        current = json.load(open(SETTINGS))
    except (OSError, ValueError):
        current = {}
    current[name] = value
    json.dump(current, open(SETTINGS, "w"), indent=2)
print("Working folder:", WORK)

## Step 2. Download the pages

This needs the **full competition data**: `train.csv`, `test.csv` and the `images/` folder with
all three collections. The class bucket (`ml-marathon-shared`) only has the Kade letters. Upload
the Kaggle download to your own bucket first (README.md, "Where the data comes from"), then type
its name and folder below. The name is remembered (outside the repository), so next time you can
leave `DATA_BUCKET` empty, and Notebooks 2 and 3 use it to save your runs.

If the files aren't where you said, the cell searches the bucket and tells you where `train.csv`
is. Images already downloaded are skipped, so it's safe to run again.

In [ ]:
import glob
import json

import boto3
import pandas as pd
from botocore.exceptions import ClientError

DATA_BUCKET = ""                 # <-- the bucket with the full competition data; empty = the one saved last time
DATA_PREFIX = "badger-scribe"    # <-- the folder in it that holds train.csv, test.csv and images/

if DATA_BUCKET:
    save_setting("bucket", DATA_BUCKET)       # remembered for next time, and for Notebooks 2 and 3
else:
    DATA_BUCKET = saved_setting("bucket")
if not DATA_BUCKET:
    raise SystemExit('Type your bucket\'s name in DATA_BUCKET first, e.g. DATA_BUCKET = "my-bucket" '
                     "(README.md, 'Where the data comes from'). It's remembered after that.")
print("Data bucket:", DATA_BUCKET)
s3 = boto3.client("s3")
images_dir = os.path.join(DATA, "images")
os.makedirs(images_dir, exist_ok=True)

def find_in_bucket(ending, most=10):
    found = []
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=DATA_BUCKET):
        found += [o["Key"] for o in page.get("Contents", []) if o["Key"].endswith(ending)]
        if len(found) >= most:
            break
    return found[:most]

def notebook_role():
    """The IAM role this notebook runs as: what a bucket policy has to name."""
    try:
        import sagemaker
        return sagemaker.get_execution_role()
    except Exception:
        pass
    try:
        arn = boto3.client("sts").get_caller_identity()["Arn"]   # arn:aws:sts::<account>:assumed-role/<role>/<session>
        account, role = arn.split(":")[4], arn.split("/")[1]
    except Exception:
        return "<this notebook's role ARN: SageMaker console → Notebook instances → this instance → IAM role ARN>"
    try:
        return boto3.client("iam").get_role(RoleName=role)["Role"]["Arn"]
    except Exception:
        return f"arn:aws:iam::{account}:role/service-role/{role}"   # most SageMaker roles live under service-role/

def bucket_policy(bucket):
    """A bucket policy letting only this notebook's role read the bucket (and save runs to it)."""
    return json.dumps({"Version": "2012-10-17", "Statement": [{
        "Effect": "Allow", "Principal": {"AWS": notebook_role()},
        "Action": ["s3:GetObject", "s3:PutObject", "s3:ListBucket"],
        "Resource": [f"arn:aws:s3:::{bucket}", f"arn:aws:s3:::{bucket}/*"]}]}, indent=2, ensure_ascii=False)

def explain(error, key):
    code = error.response.get("Error", {}).get("Code", "")
    if code in ("404", "NoSuchKey", "NotFound"):
        try:
            hits = find_in_bucket("train.csv")
        except ClientError:
            hits = None
        if hits:
            folders = sorted({h.rsplit("/", 1)[0] if "/" in h else "" for h in hits})
            return (f"There is no {key} in {DATA_BUCKET}. train.csv is in: " + ", ".join(repr(f) for f in folders)
                    + ". Set DATA_PREFIX to one of these and run again.")
        if hits == []:
            return f"The bucket {DATA_BUCKET} has no train.csv anywhere. Upload the data first (README.md)."
        return f"There is no {key} in {DATA_BUCKET}, and this notebook isn't allowed to look inside it."
    if code in ("403", "AccessDenied"):
        return (f"This notebook isn't allowed to read {DATA_BUCKET}. The bucket can stay private: give this "
                f"notebook's role access with a bucket policy. In the S3 console, open {DATA_BUCKET} → Permissions → "
                "Bucket policy → Edit, paste the policy below (or add its statement to the policy already there), "
                "Save changes, and run this cell again. It lets only this notebook's role in.\n\n"
                + bucket_policy(DATA_BUCKET))
    if code == "NoSuchBucket":
        return f"There is no bucket called {DATA_BUCKET}. Check the name in the S3 console."
    return f"S3 said: {error}"

prefix = f"{DATA_PREFIX}/" if DATA_PREFIX else ""
for name in ("train.csv", "test.csv"):
    try:
        s3.download_file(DATA_BUCKET, prefix + name, os.path.join(DATA, name))
    except ClientError as e:
        raise SystemExit(explain(e, prefix + name))
for name in ("metadata.csv", "sample_submission.csv"):   # useful, but not required
    try:
        s3.download_file(DATA_BUCKET, prefix + name, os.path.join(DATA, name))
    except ClientError:
        print(f"(no {name}; that's fine)")

train = pd.read_csv(os.path.join(DATA, "train.csv"), keep_default_na=False, encoding="utf-8-sig")
test = pd.read_csv(os.path.join(DATA, "test.csv"), keep_default_na=False, encoding="utf-8-sig")
wanted = list(pd.concat([train, test])["page_id"].drop_duplicates())
new, missing = 0, []
for i, page_id in enumerate(wanted, 1):
    path = os.path.join(images_dir, page_id + ".jpg")
    if not os.path.exists(path):
        try:
            s3.download_file(DATA_BUCKET, f"{prefix}images/{page_id}.jpg", path)
            new += 1
        except ClientError:
            missing.append(page_id)
    if i % 100 == 0:
        print(f"  {i}/{len(wanted)} pages checked", flush=True)
print(f"{len(train)} training pages, {len(test)} test pages; {new} images downloaded now, "
      f"{len(glob.glob(os.path.join(images_dir, '*.jpg')))} in {images_dir}")
if missing:
    hits = find_in_bucket(missing[0] + ".jpg", most=1)
    where = f" It is at {hits[0]!r}, so the images folder is somewhere else." if hits else ""
    print(f"WARNING: {len(missing)} images aren't at {prefix}images/<page>.jpg, e.g. {missing[0]}.{where}")

## Step 3. Prepare the pages for training

This decides how each page is read (whole, or a survey spread as two halves) and pairs every
training image with its text. It takes a few minutes, mostly finding the fold in the survey
spreads.

**Read the table it prints.** Every collection should have training, dev and test pages. If a
collection shows 0 training pages, `train.csv` isn't the full competition data.

In [ ]:
import prepare_data

items = prepare_data.build(DATA, "dev_ids.csv")

## Step 4. Check a survey spread by eye

The fold is found automatically, as the strip in the middle with the least writing. This shows a
spread with a **red line where it was cut**, then the two halves, with the start of the text each
half is trained on. The red line should run down the fold, between the two pages, not through
writing. Try a few spreads by changing `SPREAD`.

In [ ]:
from IPython.display import display
from PIL import Image, ImageDraw

parts = pd.read_csv(os.path.join(DATA, "parts.csv"), keep_default_na=False)
halves = parts[parts.part != "page"]
if halves.empty:
    print("No survey spreads were cut.")
else:
    SPREAD = halves.page_id.iloc[0]   # or type a page name, e.g. "survey_002_p0010"
    rows = parts[parts.page_id == SPREAD]
    img = Image.open(os.path.join(DATA, rows.image.iloc[0])).convert("RGB")
    left = [int(v) for v in rows[rows.part == "left"].box.iloc[0].split(",")]
    right = [int(v) for v in rows[rows.part == "right"].box.iloc[0].split(",")]
    fold = (left[2] + right[0]) // 2
    marked = img.copy()
    ImageDraw.Draw(marked).line([(fold, 0), (fold, img.height)], fill="red", width=max(3, img.width // 300))
    marked.thumbnail((900, 700))
    print(SPREAD)
    display(marked)
    texts = items[items.page_id == SPREAD]
    for name, b in (("left", left), ("right", right)):
        half = img.crop(b)
        half.thumbnail((450, 600))
        print(f"{name} half")
        display(half)
        match = texts[texts.box == ",".join(map(str, b))]
        if len(match):
            print("  trained on:", match.text.iloc[0][:120].replace("\n", " | "), "...")
    if SPREAD in set(items.page_id) and texts.box.eq("").all():
        print("This spread's label doesn't split at a page number, so it's trained on whole.")

## Step 5. See what the model trains on

Training uses each image several times, each time with small random changes, so a few pages go
further and the model gets used to the kinds of scans the dev-set audit found hard:

- every collection: a slight tilt, and small changes in brightness, contrast and sharpness;
- **Kade**: sometimes shrunk and enlarged again, and faded, like the small pencil postcards;
- **Dominy**: sometimes a faint mirror image of the page on top, like ink from the back showing through.

This shows one training page per collection as it is, then three random versions. The changes
should look like other scans of the same page, never like damage that hides the writing.

In [ ]:
import random
from common import augment, load_image

rng = random.Random(0)
for cat in ("kade_letters", "survey_notes", "dominy_accounts"):
    rows = items[items.category == cat]
    if rows.empty:
        continue
    r = rows.iloc[0]
    box = tuple(int(v) for v in r.box.split(",")) if r.box else None
    original = load_image(os.path.join(DATA, r.image), box)
    versions = [original] + [augment(original, rng, cat) for _ in range(3)]
    w = 260
    versions = [v.resize((w, int(v.height * w / v.width))) for v in versions]
    sheet = Image.new("RGB", (4 * w + 30, versions[0].height), "white")
    for i, v in enumerate(versions):
        sheet.paste(v, (i * (w + 10), 0))
    print(f"{cat}: {r.page_id} (left: as it is)")
    display(sheet)

**Done.** Go on to **Notebook 2** (`2_finetune.ipynb`).

To save the prepared data for teammates, copy the working folder to your bucket in a cell:
`!aws s3 sync {DATA} s3://<your bucket>/churro-work/data --quiet`